# Retail + Axio Channel Report

Run this notebook to regenerate `final_channel_report.xlsx` from `axio.csv`, `retail copy.csv`, and the Retail + Axio sheet in `Master April'26.xlsb`.

Rows are joined to the Retail + Axio master by `Retailer DMS id`, then grouped by the master `State`, `Dist Name`, and `Outlet Name` so AXIO/Retail values land against the correct store.

In [1]:
from pathlib import Path

import pandas as pd

from channel_report_generator import (
    AXIO_FILE,
    OUTPUT_FILE,
    RETAIL_FILE,
    build_detail_report,
    build_final_rows,
    prepare_channel_frame,
    read_master_lookup,
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)

In [2]:
master_lookup = read_master_lookup()

axio = prepare_channel_frame(AXIO_FILE, "AXIO", master_lookup)
retail = prepare_channel_frame(RETAIL_FILE, "Retail", master_lookup)

combined = pd.concat([axio, retail], ignore_index=True)
detail_report = build_detail_report(combined)
final_channel_report = pd.DataFrame(build_final_rows(detail_report))

final_channel_report.to_excel(OUTPUT_FILE, index=False)

print(f"Generated {OUTPUT_FILE.name}")
final_channel_report.tail(10)

Generated final_channel_report.xlsx


,State,DistributorName,RetailerName,AXIO Unit,AXIO GWP,Retail Unit,Retail GWP,Total Unit,Total GWP
3006,,,JAI BALAJI MOBILE (Mi),2,2298,,,2,2298
3007,,,R K ELECTRONICS (Mi),,,1,749,1,749
3008,,,SALUJA MOBILES (Mi Store),,,1,1699,1,1699
3009,,,Shree Ganesh Enterprise (Mi),,,1,949,1,949
3010,,,Shree Shree Infocom (Mi),1,1599,,,1,1599
3011,,Videotech Electronics Total,,17,21793,5,5995,22,27788
3012,,Xiaomi India Technology Pvt Ltd,Mi Home Diamond Plaza Kolkata,1,2099,,,1,2099
3013,,Xiaomi India Technology Pvt Ltd Total,,1,2099,,,1,2099
3014,West Bengal Total,,,235,356315,53,70147,288,426462
3015,Grand Total,,,5902,9139788,788,1323672,6690,10463460


In [3]:
grand_total = final_channel_report.loc[
    final_channel_report["State"].eq("Grand Total"),
    ["AXIO Unit", "AXIO GWP", "Retail Unit", "Retail GWP", "Total Unit", "Total GWP"],
].iloc[0]

source_axio = pd.read_csv(AXIO_FILE)
source_retail = pd.read_csv(RETAIL_FILE)
source_retail = source_retail[
    source_retail["Status"].eq(1)
    & source_retail["Payment Status"].astype(str).str.upper().eq("TRUE")
]

expected = pd.Series(
    {
        "AXIO Unit": len(source_axio),
        "AXIO GWP": int(round(pd.to_numeric(source_axio[" Customer Price"], errors="coerce").fillna(0).sum())),
        "Retail Unit": len(source_retail),
        "Retail GWP": int(round(pd.to_numeric(source_retail[" Customer Price"], errors="coerce").fillna(0).sum())),
        "Total Unit": len(source_axio) + len(source_retail),
        "Total GWP": int(round(
            pd.to_numeric(source_axio[" Customer Price"], errors="coerce").fillna(0).sum()
            + pd.to_numeric(source_retail[" Customer Price"], errors="coerce").fillna(0).sum()
        )),
    }
)

actual = grand_total.astype(int)
validation = pd.DataFrame({"Expected": expected, "Actual": actual, "Match": expected.eq(actual)})
validation

,Expected,Actual,Match
AXIO Unit,5902,5902,True
AXIO GWP,9139788,9139788,True
Retail Unit,788,788,True
Retail GWP,1323672,1323672,True
Total Unit,6690,6690,True
Total GWP,10463460,10463460,True


In [4]:
invalid_state_rows = final_channel_report[
    final_channel_report["State"].isin(["India Total", "Blank Total"])
]

numeric_columns = ["AXIO Unit", "AXIO GWP", "Retail Unit", "Retail GWP", "Total Unit", "Total GWP"]
check_frame = final_channel_report.fillna("").copy()
for column in numeric_columns:
    check_frame[column] = pd.to_numeric(check_frame[column], errors="coerce").fillna(0)

state_subtotal_errors = []
running_totals = {column: 0 for column in numeric_columns}
current_state = None

for row_number, row in check_frame.iterrows():
    state = str(row["State"])
    distributor = str(row["DistributorName"])

    if state and not state.endswith("Total") and state != "Grand Total":
        current_state = state
        running_totals = {column: 0 for column in numeric_columns}

    if state.endswith(" Total") and state != "Grand Total":
        mismatched_columns = [
            column for column in numeric_columns if running_totals[column] != row[column]
        ]
        if mismatched_columns:
            state_subtotal_errors.append(
                {
                    "Excel Row": row_number + 2,
                    "State": current_state,
                    "Columns": ", ".join(mismatched_columns),
                }
            )
        running_totals = {column: 0 for column in numeric_columns}
        current_state = None
    elif state != "Grand Total" and not distributor.endswith("Total"):
        for column in numeric_columns:
            running_totals[column] += row[column]

print(f"Invalid state total rows: {len(invalid_state_rows)}")
print(f"State subtotal errors: {len(state_subtotal_errors)}")

pd.DataFrame(state_subtotal_errors)

Invalid state total rows: 0
State subtotal errors: 0


""
